In [0]:
from pyspark.sql import functions as F


def ingest_csv_autoloader(
    source_dir,
    file_name,
    target_table,
    schema,
    checkpoint_path,
    batch_id,
    multiline=False,
    partition_by=None
):
    """
    Bronze CSV ingestion using Databricks Auto Loader.

    - Structured Streaming / cloudFiles
    - one checkpoint per source
    - availableNow processes all currently available files and stops
    - adds Bronze lineage metadata
    - optional partitioning for large tables
    """

    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("pathGlobFilter", file_name)
        .option("mode", "PERMISSIVE")
        .schema(schema)
    )

    if multiline:
        reader = (
            reader
            .option("multiLine", "true")
            .option("quote", '"')
            .option("escape", '"')
        )

    df = (
        reader
        .load(source_dir)
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("batch_id", F.lit(batch_id))
    )

    writer = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
    )

    if partition_by:
        writer = writer.partitionBy(partition_by)

    query = writer.toTable(target_table)

    query.awaitTermination()

    print(f"Auto Loader completed: {target_table}")